In [17]:
import pandas as pd
import numpy as np

path = "ETAPA 2 - GEOREFERENCIACIÓN DE DATA/outputs/censo_georeferencia_armenia.csv"

df_arm = pd.read_csv(path, low_memory=False)

print(df_arm.shape)
df_arm.head()

(269280, 48)


,COD_DANE_ANM,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,P_ENFERMO,...,CPOB_CCDGO,COD_SECC,MANZ_CCNCT,COD_AG,SHAPE_Leng,SHAPE_Area,MANZ_CCDGO,DPTO_MPIO_y,GEOM_WKT,DPTO_MPIO
0,6300110000000000010115,4.0,3.0,2.0,1.0,2.0,1.0,1.0,13,1.0,...,63001000,63001100000000000101,6300110000000000010115,715240,0.002082,1.382203e-07,15,63001,"POLYGON ((-75.711004 4.51884, -75.711019 4.518...",63001
1,6300110000000000010115,4.0,3.0,2.0,NaN,2.0,NaN,1.0,13,2.0,...,63001000,63001100000000000101,6300110000000000010115,715240,0.002082,1.382203e-07,15,63001,"POLYGON ((-75.711004 4.51884, -75.711019 4.518...",63001
2,6300110000000000010115,4.0,3.0,2.0,NaN,1.0,NaN,1.0,6,2.0,...,63001000,63001100000000000101,6300110000000000010115,715240,0.002082,1.382203e-07,15,63001,"POLYGON ((-75.711004 4.51884, -75.711019 4.518...",63001
3,6300110000000000010115,4.0,3.0,2.0,NaN,1.0,NaN,1.0,3,2.0,...,63001000,63001100000000000101,6300110000000000010115,715240,0.002082,1.382203e-07,15,63001,"POLYGON ((-75.711004 4.51884, -75.711019 4.518...",63001
4,6300110000000000010115,5.0,3.0,2.0,NaN,2.0,NaN,1.0,7,2.0,...,63001000,63001100000000000101,6300110000000000010115,715240,0.002082,1.382203e-07,15,63001,"POLYGON ((-75.711004 4.51884, -75.711019 4.518...",63001


In [18]:
# Servicios
df_arm["CAR_ACUEDUCTO"] = (df_arm["VB_ACU"] == 2).astype(int)
df_arm["CAR_ALCANT"] = (df_arm["VC_ALC"] == 2).astype(int)
df_arm["CAR_GAS"] = (df_arm["VD_GAS"] == 2).astype(int)
df_arm["CAR_BASURAS"] = (df_arm["VE_RECBAS"] == 2).astype(int)
df_arm["CAR_INTERNET"] = (df_arm["VF_INTERNET"] == 2).astype(int)

# Educación
df_arm["CAR_ANALFABETISMO"] = (df_arm["P_ALFABETA"] == 2).astype(int)
df_arm["CAR_BAJA_ESCOLARIDAD"] = (df_arm["P_NIVEL_ANOSR"] < 6).astype(int)

# Trabajo
df_arm["CAR_DESEMPLEO"] = (df_arm["P_TRABAJO"] == 2).astype(int)

# Hacinamiento
df_arm["HACINAMIENTO"] = df_arm["HA_TOT_PER"] / df_arm["H_NRO_DORMIT"].replace(0, np.nan)
df_arm["CAR_HACINAMIENTO"] = (df_arm["HACINAMIENTO"] > 3).astype(int)

# Materiales
MAT_PRECARIO = [5,6,7,8,9]

df_arm["CAR_PARED"] = df_arm["V_MAT_PARED"].isin(MAT_PRECARIO).astype(int)
df_arm["CAR_PISO"] = df_arm["V_MAT_PISO"].isin(MAT_PRECARIO).astype(int)

# Tipo vivienda
TIPO_PRECARIO = [3,4]

df_arm["CAR_TIPO_VIV"] = df_arm["V_TIPO_VIV"].isin(TIPO_PRECARIO).astype(int)

In [19]:
# Servicios
df_arm["IND_SERVICIOS"] = df_arm[
    ["CAR_ACUEDUCTO","CAR_ALCANT","CAR_GAS","CAR_BASURAS","CAR_INTERNET"]
].mean(axis=1)

# Educación
df_arm["IND_EDUCACION"] = df_arm[
    ["CAR_ANALFABETISMO","CAR_BAJA_ESCOLARIDAD"]
].mean(axis=1)

# Laboral
df_arm["IND_LABORAL"] = df_arm["CAR_DESEMPLEO"]

# Habitacional
df_arm["IND_POBREZA_HAB"] = df_arm[
    ["CAR_HACINAMIENTO","CAR_PARED","CAR_PISO","CAR_TIPO_VIV"]
].mean(axis=1)

# Multidimensional
df_arm["IND_MULTIDIM"] = df_arm[
    ["IND_SERVICIOS","IND_EDUCACION","IND_LABORAL","IND_POBREZA_HAB"]
].mean(axis=1)

In [20]:
armenia_manzana = (
    df_arm
    .groupby("COD_DANE_ANM")
    .agg({
        "IND_SERVICIOS":"mean",
        "IND_EDUCACION":"mean",
        "IND_LABORAL":"mean",
        "IND_POBREZA_HAB":"mean",
        "IND_MULTIDIM":"mean",
        "HA_TOT_PER":"sum"
    })
    .reset_index()
)

print(armenia_manzana.shape)
armenia_manzana.head()

(3190, 7)


,COD_DANE_ANM,IND_SERVICIOS,IND_EDUCACION,IND_LABORAL,IND_POBREZA_HAB,IND_MULTIDIM,HA_TOT_PER
0,6300110000000000010113,0.118919,0.405405,0.0,0.003378,0.131926,276.0
1,6300110000000000010114,0.101408,0.387324,0.0,0.024648,0.128345,251.0
2,6300110000000000010115,0.103226,0.379032,0.0,0.026210,0.127117,448.0
3,6300110000000000010116,0.098734,0.405063,0.0,0.000000,0.125949,221.0
4,6300110000000000010117,0.123256,0.389535,0.0,0.000000,0.128198,304.0


In [21]:
import os

OUTPUT_DIR = "ETAPA 3 - CREACIÓN DE INDICADORES/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

armenia_manzana.to_csv(
    f"{OUTPUT_DIR}/indices_manzana_armenia.csv",
    index=False
)

print("Archivo Armenia guardado")

Archivo Armenia guardado


In [22]:
import pandas as pd

df_cali = pd.read_csv(
    "ETAPA 2 - GEOREFERENCIACIÓN DE DATA/outputs/censo_georeferencia_cali.csv",
    low_memory=False
)

print(df_cali.shape)

(1802659, 48)


In [23]:
import numpy as np

# Servicios
df_cali["CAR_ACUEDUCTO"] = (df_cali["VB_ACU"] == 2).astype(int)
df_cali["CAR_ALCANT"] = (df_cali["VC_ALC"] == 2).astype(int)
df_cali["CAR_GAS"] = (df_cali["VD_GAS"] == 2).astype(int)
df_cali["CAR_BASURAS"] = (df_cali["VE_RECBAS"] == 2).astype(int)
df_cali["CAR_INTERNET"] = (df_cali["VF_INTERNET"] == 2).astype(int)

# Educación
df_cali["CAR_ANALFABETISMO"] = (df_cali["P_ALFABETA"] == 2).astype(int)
df_cali["CAR_BAJA_ESCOLARIDAD"] = (df_cali["P_NIVEL_ANOSR"] < 6).astype(int)

# Laboral
df_cali["CAR_DESEMPLEO"] = (df_cali["P_TRABAJO"] == 2).astype(int)

# Hacinamiento
df_cali["HACINAMIENTO"] = df_cali["HA_TOT_PER"] / df_cali["H_NRO_DORMIT"].replace(0, np.nan)
df_cali["CAR_HACINAMIENTO"] = (df_cali["HACINAMIENTO"] > 3).astype(int)

# Materiales precarios
MAT_PRECARIO = [5,6,7,8,9]

df_cali["CAR_PARED"] = df_cali["V_MAT_PARED"].isin(MAT_PRECARIO).astype(int)
df_cali["CAR_PISO"] = df_cali["V_MAT_PISO"].isin(MAT_PRECARIO).astype(int)

# Tipo de vivienda precaria
TIPO_PRECARIO = [3,4]

df_cali["CAR_TIPO_VIV"] = df_cali["V_TIPO_VIV"].isin(TIPO_PRECARIO).astype(int)

In [24]:
# Servicios
df_cali["IND_SERVICIOS"] = df_cali[
    ["CAR_ACUEDUCTO","CAR_ALCANT","CAR_GAS","CAR_BASURAS","CAR_INTERNET"]
].mean(axis=1)

# Educación
df_cali["IND_EDUCACION"] = df_cali[
    ["CAR_ANALFABETISMO","CAR_BAJA_ESCOLARIDAD"]
].mean(axis=1)

# Laboral
df_cali["IND_LABORAL"] = df_cali["CAR_DESEMPLEO"]

# Habitacional
df_cali["IND_POBREZA_HAB"] = df_cali[
    ["CAR_HACINAMIENTO","CAR_PARED","CAR_PISO","CAR_TIPO_VIV"]
].mean(axis=1)

# Multidimensional
df_cali["IND_MULTIDIM"] = df_cali[
    ["IND_SERVICIOS","IND_EDUCACION","IND_LABORAL","IND_POBREZA_HAB"]
].mean(axis=1)

In [25]:
cali_manzana = (
    df_cali
    .groupby("COD_DANE_ANM")
    .agg({
        "IND_SERVICIOS":"mean",
        "IND_EDUCACION":"mean",
        "IND_LABORAL":"mean",
        "IND_POBREZA_HAB":"mean",
        "IND_MULTIDIM":"mean",
        "HA_TOT_PER":"sum"
    })
    .reset_index()
)

print(cali_manzana.shape)
cali_manzana.head()

(13617, 7)


,COD_DANE_ANM,IND_SERVICIOS,IND_EDUCACION,IND_LABORAL,IND_POBREZA_HAB,IND_MULTIDIM,HA_TOT_PER
0,7600110000000001010101,0.074797,0.361789,0.008130,0.024390,0.117276,888.0
1,7600110000000001010102,0.133333,0.433333,0.000000,0.025000,0.147917,296.0
2,7600110000000001010111,0.112169,0.375661,0.005291,0.018519,0.127910,689.0
3,7600110000000001010112,0.080156,0.381323,0.000000,0.007782,0.117315,1013.0
4,7600110000000001010113,0.068696,0.386957,0.000000,0.006522,0.115543,728.0


In [26]:
OUTPUT_DIR = "ETAPA 3 - CREACIÓN DE INDICADORES/outputs"

cali_manzana.to_csv(
    f"{OUTPUT_DIR}/indices_manzana_cali.csv",
    index=False
)

print("Archivo Cali guardado")

Archivo Cali guardado


In [27]:
import pandas as pd

df_per = pd.read_csv(
    "ETAPA 2 - GEOREFERENCIACIÓN DE DATA/outputs/censo_georeferencia_pereira.csv",
    low_memory=False
)

print(df_per.shape)
df_per.head()

(377436, 48)


,COD_DANE_ANM,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,P_ENFERMO,...,CPOB_CCDGO,COD_SECC,MANZ_CCNCT,COD_AG,SHAPE_Leng,SHAPE_Area,MANZ_CCDGO,DPTO_MPIO_y,GEOM_WKT,DPTO_MPIO
0,6600110000000007010802,4.0,5.0,3.0,NaN,2.0,NaN,1.0,9,2.0,...,66001000,66001100000000070108,6600110000000007010802,725576,0.001813,8.149646e-08,2,66001,"POLYGON ((-75.704191 4.800833, -75.704246 4.80...",66001
1,6600110000000007010802,4.0,5.0,3.0,NaN,1.0,NaN,1.0,2,2.0,...,66001000,66001100000000070108,6600110000000007010802,725576,0.001813,8.149646e-08,2,66001,"POLYGON ((-75.704191 4.800833, -75.704246 4.80...",66001
2,6600110000000007010802,4.0,5.0,3.0,NaN,2.0,NaN,1.0,6,2.0,...,66001000,66001100000000070108,6600110000000007010802,725576,0.001813,8.149646e-08,2,66001,"POLYGON ((-75.704191 4.800833, -75.704246 4.80...",66001
3,6600110000000007010802,4.0,5.0,3.0,NaN,NaN,NaN,NaN,1,2.0,...,66001000,66001100000000070108,6600110000000007010802,725576,0.001813,8.149646e-08,2,66001,"POLYGON ((-75.704191 4.800833, -75.704246 4.80...",66001
4,6600110000000004010211,2.0,5.0,2.0,NaN,2.0,NaN,1.0,14,2.0,...,66001000,66001100000000040102,6600110000000004010211,724882,0.001927,2.141440e-07,11,66001,"POLYGON ((-75.687199 4.816517, -75.687199 4.81...",66001


In [28]:
# Servicios
df_per["CAR_ACUEDUCTO"] = (df_per["VB_ACU"] == 2).astype(int)
df_per["CAR_ALCANT"] = (df_per["VC_ALC"] == 2).astype(int)
df_per["CAR_GAS"] = (df_per["VD_GAS"] == 2).astype(int)
df_per["CAR_BASURAS"] = (df_per["VE_RECBAS"] == 2).astype(int)
df_per["CAR_INTERNET"] = (df_per["VF_INTERNET"] == 2).astype(int)

# Educación
df_per["CAR_ANALFABETISMO"] = (df_per["P_ALFABETA"] == 2).astype(int)
df_per["CAR_BAJA_ESCOLARIDAD"] = (df_per["P_NIVEL_ANOSR"] < 6).astype(int)

# Laboral
df_per["CAR_DESEMPLEO"] = (df_per["P_TRABAJO"] == 2).astype(int)

# Hacinamiento
df_per["HACINAMIENTO"] = df_per["HA_TOT_PER"] / df_per["H_NRO_DORMIT"].replace(0, np.nan)
df_per["CAR_HACINAMIENTO"] = (df_per["HACINAMIENTO"] > 3).astype(int)

# Materiales
MAT_PRECARIO = [5,6,7,8,9]

df_per["CAR_PARED"] = df_per["V_MAT_PARED"].isin(MAT_PRECARIO).astype(int)
df_per["CAR_PISO"] = df_per["V_MAT_PISO"].isin(MAT_PRECARIO).astype(int)

# Tipo vivienda
TIPO_PRECARIO = [3,4]

df_per["CAR_TIPO_VIV"] = df_per["V_TIPO_VIV"].isin(TIPO_PRECARIO).astype(int)

In [29]:
df_per["IND_SERVICIOS"] = df_per[
    ["CAR_ACUEDUCTO","CAR_ALCANT","CAR_GAS","CAR_BASURAS","CAR_INTERNET"]
].mean(axis=1)

df_per["IND_EDUCACION"] = df_per[
    ["CAR_ANALFABETISMO","CAR_BAJA_ESCOLARIDAD"]
].mean(axis=1)

df_per["IND_LABORAL"] = df_per["CAR_DESEMPLEO"]

df_per["IND_POBREZA_HAB"] = df_per[
    ["CAR_HACINAMIENTO","CAR_PARED","CAR_PISO","CAR_TIPO_VIV"]
].mean(axis=1)

df_per["IND_MULTIDIM"] = df_per[
    ["IND_SERVICIOS","IND_EDUCACION","IND_LABORAL","IND_POBREZA_HAB"]
].mean(axis=1)

In [30]:
pereira_manzana = (
    df_per
    .groupby("COD_DANE_ANM")
    .agg({
        "IND_SERVICIOS":"mean",
        "IND_EDUCACION":"mean",
        "IND_LABORAL":"mean",
        "IND_POBREZA_HAB":"mean",
        "IND_MULTIDIM":"mean",
        "HA_TOT_PER":"sum"
    })
    .reset_index()
)

print(pereira_manzana.shape)
pereira_manzana.head()

(3988, 7)


,COD_DANE_ANM,IND_SERVICIOS,IND_EDUCACION,IND_LABORAL,IND_POBREZA_HAB,IND_MULTIDIM,HA_TOT_PER
0,6600110000000001010101,0.253333,0.166667,0.0,0.033333,0.113333,43.0
1,6600110000000001010105,0.057718,0.402685,0.0,0.028523,0.122232,571.0
2,6600110000000001010106,0.045283,0.433962,0.0,0.018868,0.124528,167.0
3,6600110000000001010109,0.000000,0.333333,0.0,0.000000,0.083333,69.0
4,6600110000000001010110,0.049412,0.317647,0.0,0.000000,0.091765,325.0


In [31]:
import os

OUTPUT_DIR = "ETAPA 3 - CREACIÓN DE INDICADORES/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pereira_manzana.to_csv(
    f"{OUTPUT_DIR}/indices_manzana_pereira.csv",
    index=False
)

print("Archivo Pereira guardado")

Archivo Pereira guardado
